# GRM: HapMap3 panel QC'd within the final set, sharded with plink 1.9

In [ ]:
RUN_ID = "sim"
FORCE = False

In [ ]:
import json, os, subprocess, sys, tempfile
REPO = os.getcwd()
while not os.path.exists(os.path.join(REPO, "pyproject.toml")):
    REPO = os.path.dirname(REPO)
sys.path[:0] = [REPO, os.path.join(REPO, "src")]
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from aoucov import load_run, repo_root
from aoucov.batch import run_script
from aoucov.manifest import needs_run, require_ready, single_target, target_items, stage_status
from aoucov.provenance import collect_meta, record, write_meta
from aoucov.report import Report
from aoucov.store import get_store

cfg = load_run(RUN_ID)
THREADS = cfg.params.get("threads", 4)

In [ ]:
from stage_helpers import verify_panel
from aoucov.runner import run_stage
require_ready(cfg, "grm_panel")
rep = Report(cfg, "grm_panel")
OUT = single_target(cfg, "grm_panel")
PREFIX = OUT[:-4]
geno_key = "acaf_bfile" if "acaf_bfile" in cfg.params["inputs"] else "acaf_pfile"
panel_env = dict(PLINK2=cfg.tool("plink2"), PLINK=cfg.tool("plink"),
                 GENO_FLAG="--bfile" if geno_key == "acaf_bfile" else "--pfile",
                 GENO_PATTERN=cfg.input(geno_key, chr="{chr}"), NCHR=22,
                 EXTRACT=cfg.input("hm3_ids"), MAF=cfg.param("qc_maf"), HWE=cfg.param("qc_hwe"),
                 GENO=cfg.param("qc_geno"), THREADS=THREADS)

In [ ]:
if needs_run(OUT, FORCE):
    with record(cfg, "grm_panel", OUT, tools={"plink2": cfg.tool("plink2"), "plink": cfg.tool("plink")}):
        run_script(cfg, "panel.sh", {**panel_env, "KEEP": single_target(cfg, "ancestry_round2"),
                                     "DROP_AMBIGUOUS": 0, "FREQ": 1, "OUT": PREFIX})

In [ ]:
counts = pd.read_csv(PREFIX + ".counts.tsv", sep="\t")
rep.count("n_samples", counts.set_index("step").loc["samples", "variants"])
steps = counts[counts["step"] != "samples"]
rep.table(steps, "variants_by_step")
rep.note(n_variants=int(steps["variants"].iloc[-1]))
fig, ax = plt.subplots(figsize=(6, 3))
ax.barh(steps["step"], steps["variants"]); ax.invert_yaxis(); ax.set_xlabel("variants")
rep.fig(fig, "variants_by_step"); plt.show()

In [ ]:
verify_panel(rep, PREFIX, cfg.tool("plink2"), cfg.param("qc_maf"), cfg.param("qc_geno"),
             cfg.param("qc_hwe"), "the final set", ambiguous_ok=True)
rep.check("allele frequencies computed once for every shard", os.path.exists(PREFIX + ".frq"))
rep.save()

## Shards

In [ ]:
N_SHARDS = cfg.param("n_shards")

def shard(cfg, item, uri, meta):
    run_script(cfg, "grm_shard.sh", dict(PLINK=cfg.tool("plink"), BFILE=PREFIX, K=item["i"], N=N_SHARDS,
                                         OUT=uri.split(".grm.bin.")[0], THREADS=1))

require_ready(cfg, "grm_shards")
run_stage(cfg, "grm_shards", shard, jobs=min(N_SHARDS, THREADS), force=FORCE)

In [ ]:
rep = Report(cfg, "grm_shards")
shards = [u for u, _ in target_items(cfg, "grm_shards")]
id_file = shards[0].split(".grm.bin.")[0] + ".grm.id"
n = sum(1 for _ in open(id_file))
nbytes = sum(os.path.getsize(s) for s in shards)
rep.check("shards hold exactly N(N+1)/2 entries", nbytes == 4 * n * (n + 1) // 2, f"{nbytes:,} bytes")
rep.check("GRM N equals the final set", n == sum(1 for _ in open(single_target(cfg, "ancestry_round2"))))
if nbytes < 2e9:
    tri = np.concatenate([np.fromfile(s, dtype="float32") for s in shards])
    diag = tri[np.arange(1, n + 1) * (np.arange(1, n + 1) + 1) // 2 - 1]
    rep.note(diag_mean=round(float(diag.mean()), 4), offdiag_mean=round(float((tri.sum() - diag.sum()) / (len(tri) - n)), 5))
    rep.check("GRM diagonal mean ~ 1", abs(diag.mean() - 1) < 0.05, f"{diag.mean():.3f}")
rep.save()